# 03 - Baselines

Before fitting anything clever, set the bar with models that know almost nothing.
Anything later has to beat these on the same data.

**Target:** `delta_s`, a lap's time minus the driver's own median clean lap in that
race (see `f1lab/features.py`). This removes car, driver and track pace, leaving tyre,
fuel and conditions.

**Evaluation:** train on 2022-2023, score on 2024. The 2025 season is held back for
one final evaluation at the end and is not touched here. Twelve wet or drying races are
excluded by two rules applied to every season (a pace rule and a wet-tyre rule, see
`f1lab/features.py`).

In [1]:
import numpy as np
import pandas as pd
from f1lab.features import prepare, split, mae, rmse, drying_races, wet_races

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
print("excluded wet/drying races (season, round):", sorted(drying_races(clean) | wet_races(raw)))
df = prepare(raw, clean)
train, val = split(df, "dev")
print(f"{len(df)} laps | train {len(train)} (2022-2023) | validation {len(val)} (2024)")
unseen = ~val["track"].isin(train["track"].unique())
print("validation laps at tracks never seen in training:", int(unseen.sum()), sorted(val[unseen]["track"].unique()))
df["delta_s"].describe(percentiles=[.01, .5, .99]).round(2)

excluded wet/drying races (season, round): [(2022, 4), (2022, 7), (2022, 17), (2022, 18), (2023, 7), (2023, 14), (2024, 9), (2024, 12), (2024, 21), (2025, 1), (2025, 12), (2025, 13)]
76249 laps | train 34690 (2022-2023) | validation 20752 (2024)
validation laps at tracks never seen in training: 3130 ['Imola', 'Monte Carlo', 'Shanghai']


count    76249.00
mean         0.07
std          1.39
min         -6.24
1%          -2.45
50%          0.00
99%          3.33
max         31.53
Name: delta_s, dtype: float64

The target has a heavy tail: it runs from about -6 s to over +31 s. These are mostly
real incident laps (spins, off-track moments, unflagged yellows) from drivers with
plenty of clean laps, not artefacts of a bad base pace. Counting them:

In [2]:
big = df[df["delta_s"].abs() > 8]
print(f"laps with |delta| > 8 s: {len(big)} ({len(big) / len(df):.2%} of all laps)")
print("RMSE is dominated by these; MAE is the headline metric, RMSE secondary.")

laps with |delta| > 8 s: 245 (0.32% of all laps)
RMSE is dominated by these; MAE is the headline metric, RMSE secondary.


## The baselines

In [3]:
y = val["delta_s"]
by_compound = train.groupby("compound")["delta_s"].mean()
by_track_compound = train.groupby(["track", "compound"])["delta_s"].mean()
bins = [0, 5, 10, 15, 20, 30, 100]
by_age = train.assign(b=pd.cut(train["tyre_age"], bins)).groupby(["compound", "b"], observed=True)["delta_s"].mean()
val_b = val.assign(b=pd.cut(val["tyre_age"], bins))

preds = {
    "B0 predict 0 (no change)": pd.Series(0.0, index=val.index),
    "B1 mean by compound": val["compound"].map(by_compound),
    "B2 mean by track + compound": pd.Series(
        [by_track_compound.get((t, c), np.nan) for t, c in zip(val["track"], val["compound"])], index=val.index
    ).fillna(val["compound"].map(by_compound)),
    "B3 mean by compound + tyre-age bucket": pd.Series(
        [by_age.get((c, b), np.nan) for c, b in zip(val_b["compound"], val_b["b"])], index=val.index
    ).fillna(val["compound"].map(by_compound)),
}
results = pd.DataFrame({k: {"MAE (s)": mae(y, p), "RMSE (s)": rmse(y, p)} for k, p in preds.items()}).T.round(3)
results

,MAE (s),RMSE (s)
B0 predict 0 (no change),0.882,1.365
B1 mean by compound,0.856,1.346
B2 mean by track + compound,0.856,1.348
B3 mean by compound + tyre-age bucket,0.855,1.345


Reading the table:
- Predicting no change from the driver's median pace is off by about 0.88 s on average.
- Compound averages (B1), compound-by-tyre-age averages (B3) and compound-by-track (B2)
  all land at 0.855 to 0.856 s. None of them knows how long the race has run, and tyre
  age and track add nothing over compound alone here.
- The per-compound means are odd: MEDIUM is +0.27 s but HARD and SOFT are slightly
  negative. A likely reason, not yet tested, is *when* each compound is used. Mediums run
  early on heavy fuel, hards later on light fuel, so the compound is partly standing in
  for fuel load.

None of these baselines knows the lap number, and fuel burn moves lap time by
seconds across a race, so the obvious next step is a model with a fuel term. That is
the hypothesis the linear model will test.

*Note on the data:* an earlier version of this notebook used a weaker wet-race rule and
scored every baseline slightly worse (best 0.882). Wet and drying laps were
contaminating the target; see the wet-race rule in `features.py`.

In [4]:
print("training mean delta by compound:", by_compound.round(3).to_dict())
print(f"spread of the target on validation: mean abs deviation {(y - y.median()).abs().mean():.3f} s, std {y.std():.3f} s")

training mean delta by compound: {'HARD': -0.039, 'MEDIUM': 0.27, 'SOFT': -0.076}
spread of the target on validation: mean abs deviation 0.882 s, std 1.364 s
